# 02 - EDA + Feature Engineering

**Use case:** Cancer-wise Drug Intelligence  
**Input:** `data/raw/oncology_trials_flat.csv` and `data/raw/oncology_interventions.csv`  
**Outputs:**
- `data/processed/trials_processed.csv`
- `data/processed/interventions_processed.csv`
- `data/processed/drug_phase_map.csv`
- `data/processed/cancer_hierarchy_final.json`
- `data/processed/cancer_conditions_list_final.csv`
- `data/processed/drug_combination_partners.json`

This notebook is intentionally organized into larger rerunnable blocks. The expensive parts show explicit progress bars so you can see that work is moving.

## Pipeline Overview

1. Load models and raw data  
2. Prune columns and clean types  
3. Audit nulls and inspect awkward trial groups  
4. Classify biological interventions  
5. Analyze enrollment and phase structure  
6. Build cancer superset -> subset hierarchy  
7. Extract title-level features  
8. Extract eligibility-level features with batched NER  
9. Standardize outcomes  
10. Normalize assets and remove non-drug leakage  
11. Build combination partner and drug-phase outputs  
12. Save final processed files

In [1]:
import ast
import json
import os
import re
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import spacy
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 160)
tqdm.pandas()

PHASE_ORDER = ["Early Phase 1", "Phase 1", "Phase 2", "Phase 3", "Phase 4"]

KEEP_COLS = [
    "nct_id",
    "brief_title",
    "official_title",
    "overall_status",
    "start_date",
    "primary_completion_date",
    "completion_date",
    "first_posted_date",
    "lead_sponsor_name",
    "lead_sponsor_class",
    "collaborators",
    "collaborator_classes",
    "brief_summary",
    "conditions",
    "keywords",
    "study_type",
    "phases",
    "phase_raw",
    "allocation",
    "intervention_model",
    "primary_purpose",
    "masking",
    "enrollment_count",
    "enrollment_type",
    "num_arms",
    "arm_labels",
    "arm_types",
    "num_interventions",
    "intervention_names",
    "intervention_types",
    "drug_names",
    "drug_other_names",
    "biological_names",
    "num_biologicals",
    "primary_outcome_measures",
    "primary_outcome_timeframes",
    "secondary_outcome_measures",
    "eligibility_criteria",
    "healthy_volunteers",
    "sex",
    "minimum_age",
    "maximum_age",
    "std_ages",
    "num_locations",
    "countries",
    "num_countries",
    "fda_regulated_drug",
    "has_results",
]

PIPE_COLS = [
    "conditions",
    "countries",
    "drug_names",
    "drug_other_names",
    "biological_names",
    "arm_labels",
    "arm_types",
    "intervention_names",
    "intervention_types",
    "collaborators",
    "collaborator_classes",
]

DATE_COLS = [
    "start_date",
    "primary_completion_date",
    "completion_date",
    "first_posted_date",
]

print("Imports OK")

Imports OK


## 1. Load Models and Data

In [2]:
print("Loading en_ner_bc5cdr_md (chemicals + diseases)...")
nlp_bc5cdr = spacy.load("en_ner_bc5cdr_md")

print("Loading en_core_sci_md (general biomedical model)...")
nlp_sci = spacy.load("en_core_sci_md")

print()
print(f"bc5cdr entity types: {nlp_bc5cdr.pipe_labels['ner']}")
print(f"sci_md entity types: {nlp_sci.pipe_labels['ner']}")

Loading en_ner_bc5cdr_md (chemicals + diseases)...
Loading en_core_sci_md (general biomedical model)...

bc5cdr entity types: ['CHEMICAL', 'DISEASE']
sci_md entity types: ['ENTITY']


In [3]:
project_root = Path.cwd()
raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(exist_ok=True)

flat_path = raw_dir / "oncology_trials_flat.csv"
inv_path = raw_dir / "oncology_interventions.csv"

if not flat_path.exists() or not inv_path.exists():
    raise FileNotFoundError(
        "Expected raw files were not found.\n"
        f"Looked for:\n- {flat_path}\n- {inv_path}"
    )

df = pd.read_csv(flat_path, low_memory=False)
df_inv = pd.read_csv(inv_path, low_memory=False)

print(f"Flat CSV      : {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Interventions : {df_inv.shape[0]:,} rows x {df_inv.shape[1]} columns")

Flat CSV      : 123,526 rows x 68 columns
Interventions : 245,442 rows x 10 columns


## 2. Prune Columns and Normalize Basic Types

This block is deliberately broad enough that once it succeeds, later sections can be rerun independently without revisiting raw ingestion.

In [8]:
missing_keep = [col for col in KEEP_COLS if col not in df.columns]
if missing_keep:
    raise KeyError(f"Missing expected columns in flat CSV: {missing_keep}")

df = df[KEEP_COLS].copy()

for col in PIPE_COLS:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str).str.strip()

for col in DATE_COLS:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

if "enrollment_count" in df.columns:
    df["enrollment_count"] = pd.to_numeric(df["enrollment_count"], errors="coerce")

df["phase_raw"] = df["phase_raw"].fillna("NA").replace({"": "NA"})

def parse_pipe(value):
    if pd.isna(value):
        return []
    value = str(value).strip()
    if not value or value.lower() == "nan":
        return []
    return [item.strip() for item in value.split("|") if item.strip()]
    
def explode_pipe_counts(series: pd.Series) -> pd.Series:
    """Split pipe-delimited strings in a Series, explode, and return value counts."""
    return (
        series.fillna("")
        .astype(str)
        .str.split("|")
        .explode()
        .str.strip()
        .replace("", pd.NA)
        .dropna()
        .value_counts()
    )

print(f"Columns after pruning: {len(df.columns)}")
df.head(3)

Columns after pruning: 48


,nct_id,brief_title,official_title,overall_status,start_date,primary_completion_date,completion_date,first_posted_date,lead_sponsor_name,lead_sponsor_class,collaborators,collaborator_classes,brief_summary,conditions,keywords,study_type,phases,phase_raw,allocation,intervention_model,primary_purpose,masking,enrollment_count,enrollment_type,num_arms,arm_labels,arm_types,num_interventions,intervention_names,intervention_types,drug_names,drug_other_names,biological_names,num_biologicals,primary_outcome_measures,primary_outcome_timeframes,secondary_outcome_measures,eligibility_criteria,healthy_volunteers,sex,minimum_age,maximum_age,std_ages,num_locations,countries,num_countries,fda_regulated_drug,has_results
0,NCT07491497,A Phase 1/2 Study of TRI-611 in ALK-Positive NSCLC,"A Phase 1/2, Dose Escalation and Expansion Study of TRI-611, an Oral ALK Molecular Glue Degrader in Participants With Advanced ALK-Positive NSCLC",RECRUITING,2026-03-11,2029-05-31,2034-01-30,2026-03-24,"TRIANA Biomedicines, Inc.",INDUSTRY,,,The goal of this clinical trial is to learn about the safety and recommended dose of TRI-611 when administered to adults with ALK-positive non-small cell lu...,ALK-positive NSCLC|ALK-Positive Lung Cancer|ALK-positive Non-small Cell Lung Cancer,NaN,INTERVENTIONAL,PHASE1|PHASE2,PHASE1|PHASE2,NON_RANDOMIZED,SEQUENTIAL,TREATMENT,NONE,160.0,ESTIMATED,4,Part 1: Dose Escalation and Backfill|Part 2: Cohort M1|Part 2: Cohort M2|Part 2: Cohort M3,EXPERIMENTAL|EXPERIMENTAL|EXPERIMENTAL|EXPERIMENTAL,1,TRI-611,DRUG,TRI-611,,,0,Part 1: Treatment emergent adverse events|Part 2: Objective response rate (ORR)|Part 2: Depth of response (DofR),Within 28 days of the first TRI-611 dose|Approximately 16 weeks after the last participant dosed in Part 2|Approximately 16 weeks after the last participant...,Part 1: Half-life (t1/2) of TRI-611|Part 1: Area under the curve (AUC) of TRI-611|Part 1: Maximum plasma concentration (Cmax) of TRI-611|Part 1: Minimum pla...,Inclusion Criteria:\n\n* Pathologically confirmed diagnosis of ALK-positive non-small cell lung cancer (NSCLC)\n* Measurable disease per RECIST v1.1\n* Adeq...,False,ALL,18 Years,NaN,ADULT|OLDER_ADULT,7,United States,1,True,False
1,NCT00669097,"Absorption, Distribution, Metabolism and Excretion (ADME) Study of TKI258 in Patients With Advanced Solid Malignancies","An Open-Label, Non-Randomized, Single-Center, Phase I Study to Determine the Absorption, Distribution, Metabolism and Excretion (ADME) of TKI258 After a Sin...",COMPLETED,NaT,NaT,NaT,2008-04-29,Novartis Pharmaceuticals,INDUSTRY,,,"This is a phase l study to examine Absorption, Distribution, Metabolism and Excretion of TKI258. There are 2 cohorts. Cohort 1 (4 patients) will receive sin...",Advanced Solid Malignancies,ADME|TKI258|RTKs inhibitor|PDGRF inhibitor|VEGFER inhibitor,INTERVENTIONAL,PHASE1,PHASE1,NaN,SINGLE_GROUP,TREATMENT,NONE,13.0,ACTUAL,1,TKI258,EXPERIMENTAL,1,TKI258,DRUG,TKI258,,,0,Cohort1: Find out about the routes and rates of excretion of TKI258 and its metabolites|Cohort 2: Safety and Tolerability of TKI258,Day 15|Time to patient withdrawal due to disease progression or tolerability issues,Cohort 1: Safety and tolerability of TKI258|Cohort 1: Preliminary anti-tumor activity of TKI258|Preliminary Anti-tumor activity of TKI258,Inclusion criteria:\n\n1. Aged ≥ 18 years\n2. Patients with histologically confirmed solid tumor or lymphoma who are resistant/refractory to approved therap...,False,ALL,18 Years,NaN,ADULT|OLDER_ADULT,1,Netherlands,1,False,False
2,NCT07490990,"A Real-World, Single-Arm Study Protocol of Disitamab Vedotin in Combination With Immunotherapy and Multimodal Radiation Therapy for HER2-Positive Advanced G...","A Real-World, Single-Arm Study Protocol of Disitamab Vedotin in Combination With Immunotherapy and Multimodal Radiation Therapy for HER2-Positive Advanced G...",NOT_YET_RECRUITING,NaT,NaT,NaT,2026-03-24,West China Hospital,OTHER,Jiangsu Taizhou People's Hospital|Chengdu First People's Hospital,OTHER|O

## 3. Null Audit and Quick Diagnostics

In [9]:
nulls = df.isna().sum().sort_values(ascending=False)
nulls = nulls[nulls > 0]
print("Columns with nulls:")
display(nulls.to_frame("null_count").head(20))

print("\nStudy type distribution:")
print(df["study_type"].value_counts(dropna=False).to_string())

print("\nPhase distribution:")
print(df["phase_raw"].value_counts(dropna=False).to_string())

Columns with nulls:


,null_count
maximum_age,79368
allocation,69637
completion_date,53419
primary_completion_date,53196
phases,52467
start_date,46311
keywords,45556
secondary_outcome_measures,29848
intervention_model,29641
masking,28753



Study type distribution:
study_type
INTERVENTIONAL     97396
OBSERVATIONAL      25694
EXPANDED_ACCESS      436

Phase distribution:
phase_raw
NA               52467
PHASE2           30208
PHASE1           16598
PHASE3            9618
PHASE1|PHASE2     8571
PHASE4            2635
EARLY_PHASE1      1942
PHASE2|PHASE3     1487


## 4. Investigate Ambiguous Trial Groups

We do not silently drop awkward records. We inspect them, explain them, and only then decide what belongs inside the drug-intelligence layer.

In [10]:
na_phase = df[df["phase_raw"] == "NA"].copy()
print("NA phase breakdown by study type:")
print(na_phase["study_type"].value_counts().to_string())

na_interventional = na_phase[na_phase["study_type"] == "INTERVENTIONAL"].copy()
print(f"\nInterventional trials with NA phase: {len(na_interventional):,}")
if not na_interventional.empty:
    sample_n = min(15, len(na_interventional))
    display(
        na_interventional[["nct_id", "brief_title", "intervention_types", "primary_purpose"]]
        .sample(sample_n, random_state=42)
    )

no_drugs = df[(df["drug_names"] == "") & (df["study_type"] == "INTERVENTIONAL")].copy()
print("Interventional trials with empty drug_names:")
print(f"Count: {len(no_drugs):,}")
print()
print("Intervention type mix among these trials:")
print(explode_pipe_counts(no_drugs["intervention_types"]).head(20).to_string())

NA phase breakdown by study type:
study_type
INTERVENTIONAL     26337
OBSERVATIONAL      25694
EXPANDED_ACCESS      436

Interventional trials with NA phase: 26,337


,nct_id,brief_title,intervention_types,primary_purpose
51853,NCT05120180,Effect of ALND With Vein Branches Reservation on Postoperative Upper Limb Edema and Dysfunction in Breast Cancer,PROCEDURE|PROCEDURE,TREATMENT
22184,NCT04832113,Impact of Patient's Therapeutic Education in APA and Dietetic on Radiotherapy Reproducibility Sessions for Prostate Cancer,OTHER|OTHER,OTHER
63585,NCT07450703,The Effect of the Health Belief Model-Based Education Given to Mothers of Children Aged 9-15 on HPV Knowledge Level and Child Vaccination,BEHAVIORAL,PREVENTION
83816,NCT05059990,Low Intensity Aerobic Exercise and Active Exercises in Cancer Patients Receiving Palliative Care,OTHER|OTHER,SUPPORTIVE_CARE
108376,NCT06654466,"Closing the GAPS: Guideline Adherence, Prevention and Surveillance in Hereditary Cancer",DEVICE,SUPPORTIVE_CARE
92401,NCT02387645,LUDEC Study - Pilot Study of the Lavage of the Uterine Cavity for the Diagnosis of Endometrial Carcinoma,PROCEDURE,DIAGNOSTIC
101985,NCT04654403,Study of First-line Camrelizumab With or Without Chemotherapy for Advanced Esophageal Squamous Cell Cancer,DRUG,TREATMENT
15960,NCT01108484,"Program of Exercise, Diet and Control of Psychological Stress in Cancer Survivors",BEHAVIORAL|BEHAVIORAL|BEHAVIORAL,SUPPORTIVE_CARE
5735,NCT02396745,TECR & ECM Placement for Esophageal High Grade Dysplasia,DEVICE,TREATMENT
56866,NCT06511323,Evaluation of Brain Metastases Treated With Stereotactic Radiotherapy Using Dynamic [18F]FDG PET,DIAGNOSTIC_TEST,DIAGNOSTIC


Interventional trials with empty drug_names:
Count: 34,773

Intervention type mix among these trials:
intervention_types
OTHER                  14193
PROCEDURE              12243
BIOLOGICAL              8709
BEHAVIORAL              8567
DEVICE                  6308
RADIATION               4357
DIAGNOSTIC_TEST         2128
DIETARY_SUPPLEMENT      1649
COMBINATION_PRODUCT      817
GENETIC                  447


## 5. Biological Intervention Classification

This is the first expensive block. It uses NER on unique biological intervention names, and the progress bar shows exactly how much work is left.

In [11]:
unique_biological_names = sorted({
    item
    for value in df["biological_names"].tolist()
    for item in parse_pipe(value)
})

print(f"Unique biological names to classify: {len(unique_biological_names):,}")

def classify_biological(name: str, nlp_model) -> str:
    if not name or str(name).strip().lower() == "nan":
        return "NON_CHEMICAL"
    doc = nlp_model(str(name)[:300])
    labels = {ent.label_ for ent in doc.ents}
    return "CHEMICAL" if "CHEMICAL" in labels else "NON_CHEMICAL"

bio_classifications = {}
for name in tqdm(unique_biological_names, desc="Classifying biological names"):
    bio_classifications[name] = classify_biological(name, nlp_bc5cdr)

bio_class_counts = pd.Series(bio_classifications).value_counts()
print("\nBiological classification counts:")
print(bio_class_counts.to_string())

chemical_examples = [k for k, v in bio_classifications.items() if v == "CHEMICAL"][:25]
nonchemical_examples = [k for k, v in bio_classifications.items() if v == "NON_CHEMICAL"][:25]
print("\nCHEMICAL examples:")
print(pd.Series(chemical_examples).to_string(index=False))
print("\nNON_CHEMICAL examples:")
print(pd.Series(nonchemical_examples).to_string(index=False))

Unique biological names to classify: 8,358


Classifying biological names:   0%|          | 0/8358 [00:00<?, ?it/s]


Biological classification counts:
NON_CHEMICAL    6597
CHEMICAL        1761

CHEMICAL examples:
                                                                                      10 mg/kg TRU-016 + Rituximab
                                                                                                     111In Zevalin
                                                                                              13-cis-Retinoic Acid
1429757-68-5, HM781-36B, NOV-1201 Hydrochloride, NOV120101 Hydrochloride, Poziotinib HCl, POZIOTINIB HYDROCHLORIDE
                                          15-valent recombinant human papillomavirus vaccine (Hansenulapolymorpha)
                                                                                          19-28z/IL-18 CAR T cells
                                                                                                 19UCART injection
                    2 doses of vaccine Quadrivalent Recombinant Human Papillomavirus Vaccine (Hans

In [12]:
def get_biological_drugs(bio_str: str) -> str:
    if not bio_str or str(bio_str).strip().lower() == "nan":
        return ""
    names = [n.strip() for n in str(bio_str).split("|") if n.strip()]
    drugs_only = [n for n in names if bio_classifications.get(n, "NON_CHEMICAL") == "CHEMICAL"]
    return "|".join(drugs_only)

df["biological_drugs"] = df["biological_names"].apply(get_biological_drugs)
print(f"Trials with biological drugs extracted: {(df['biological_drugs'] != '').sum():,}")
display(df[df["biological_drugs"] != ""][['nct_id', 'biological_names', 'biological_drugs']].head(10))

Trials with biological drugs extracted: 3,244


,nct_id,biological_names,biological_drugs
20,NCT06178588,Sacituzumab Govitecan,Sacituzumab Govitecan
30,NCT03282188,Reolysin,Reolysin
33,NCT03579927,Filgrastim|Rituximab|Umbilical Cord Blood-derived Natural Killer Cells,Rituximab
60,NCT01606241,Multi-epitope Folate Receptor Alpha Peptide Vaccine,Multi-epitope Folate Receptor Alpha Peptide Vaccine
191,NCT00034554,gp75 DNA vaccine,gp75 DNA vaccine
227,NCT00911560,adjuvant OPT-821 in a vaccine containing two antigens (GD2L and GD3L) covalently linked to KLH|oral β-glucan,adjuvant OPT-821 in a vaccine containing two antigens (GD2L and GD3L) covalently linked to KLH
390,NCT01413191,Cixutumumab,Cixutumumab
499,NCT00327496,DC vaccine,DC vaccine
595,NCT01584115,NY-ESO-1 combined with MPLA|NY-ESO-1 combined with MPLA vaccine,NY-ESO-1 combined with MPLA vaccine
596,NCT05981235,AZD6422 CLDN18.2 CAR-T product,AZD6422 CLDN18.2 CAR-T product


## 6. Merge Drug Sources and Enrollment Audit

In [13]:
def merge_drug_sources(drug_str: str, bio_drug_str: str) -> str:
    all_drugs = []
    for source in [drug_str, bio_drug_str]:
        if source and str(source).strip().lower() != "nan":
            all_drugs.extend([x.strip() for x in str(source).split("|") if x.strip()])

    seen = []
    for item in all_drugs:
        if item.lower() not in [s.lower() for s in seen]:
            seen.append(item)
    return "|".join(seen)

df["all_drugs_raw"] = df.apply(lambda row: merge_drug_sources(row["drug_names"], row["biological_drugs"]), axis=1)

print(f"Trials with any drug source          : {(df['all_drugs_raw'] != '').sum():,}")
print(f"Trials with raw drug_names only      : {(df['drug_names'] != '').sum():,}")
print(f"Additional trials from biologicals   : {(df['all_drugs_raw'] != '').sum() - (df['drug_names'] != '').sum():,}")

print("\nEnrollment summary:")
display(df["enrollment_count"].describe().to_frame("enrollment_count"))

print("Top 15 enrollment outliers:")
display(
    df[["nct_id", "brief_title", "enrollment_count", "study_type", "phase_raw"]]
    .sort_values("enrollment_count", ascending=False)
    .head(15)
)

Trials with any drug source          : 66,988
Trials with raw drug_names only      : 65,358
Additional trials from biologicals   : 1,630

Enrollment summary:


,enrollment_count
count,1.213100e+05
mean,3.847710e+03
std,3.347637e+05
min,0.000000e+00
25%,2.600000e+01
50%,6.000000e+01
75%,1.740000e+02
max,1.000000e+08


Top 15 enrollment outliers:


,nct_id,brief_title,enrollment_count,study_type,phase_raw
76089,NCT01166009,CIBMTR Research Database,99999999.0,OBSERVATIONAL,NA
70098,NCT06268535,Identification of Anticancer Drugs Associated With Cancer Therapy-related Cardiac Dysfunction: a Pharmacovigilance Study,36580288.0,OBSERVATIONAL,NA
110689,NCT00904579,Cancer Risk in Organ Transplant Recipients and End-Stage Renal Disease,19929901.0,OBSERVATIONAL,NA
55727,NCT02553148,Estimating the Global Need for Palliative Care for Children,18837613.0,OBSERVATIONAL,NA
103302,NCT06403917,Evaluation of Breast Cancer Screening Program at Assiut Governorate,15000000.0,OBSERVATIONAL,NA
60104,NCT04365114,Patient Outcomes from Second Film-readers and Test Threshold Relaxation in Breast Screening,13094122.0,OBSERVATIONAL,NA
48604,NCT05291728,Screening for Early Gastric Cancer in Shaanxi Province,12500000.0,OBSERVATIONAL,NA
38608,NCT07554157,Development of a Pan-Cancer Screening Model Based on Blood Biomarkers,10000000.0,OBSERVATIONAL,NA
70455,NCT05885490,IReC-Bio and IReC Registry,10000000.0,OBSERVATIONAL,NA
39595,NCT05247463,"English Mammography Screening Outcomes by Age, Frequency and Test Threshold",10000000.0,OBSERVATIONAL,NA


In [14]:
fig_enrollment = px.histogram(
    df[df["enrollment_count"].notna() & (df["enrollment_count"] < df["enrollment_count"].quantile(0.99))],
    x="enrollment_count",
    nbins=60,
    color="study_type",
    title="Enrollment Distribution Below the 99th Percentile",
)
fig_enrollment.update_layout(height=420)
fig_enrollment.show()

## 7. Phase Bucketing and Observational Context

In [15]:
def phase_to_buckets(phase_raw: str) -> list:
    value = str(phase_raw).strip().upper()
    if value in {"", "NA", "NAN"}:
        return []
    if value == "EARLY_PHASE1":
        return ["Early Phase 1"]
    if value == "PHASE1":
        return ["Phase 1"]
    if value == "PHASE2":
        return ["Phase 2"]
    if value == "PHASE3":
        return ["Phase 3"]
    if value == "PHASE4":
        return ["Phase 4"]
    if value == "PHASE1|PHASE2":
        return ["Phase 1", "Phase 2"]
    if value == "PHASE2|PHASE3":
        return ["Phase 2", "Phase 3"]
    return []

df["phase_buckets"] = df["phase_raw"].apply(phase_to_buckets)

all_buckets = [bucket for buckets in df["phase_buckets"] for bucket in buckets]
print("Phase bucket distribution:")
print(pd.Series(all_buckets).value_counts().reindex(PHASE_ORDER).fillna(0).astype(int).to_string())

df_obs = df[df["study_type"] == "OBSERVATIONAL"].copy()
df_interventional = df[df["study_type"] == "INTERVENTIONAL"].copy()
print()
print(f"Interventional trials : {len(df_interventional):,}")
print(f"Observational trials  : {len(df_obs):,}")

Phase bucket distribution:
Early Phase 1     1942
Phase 1          25169
Phase 2          40266
Phase 3          11105
Phase 4           2635

Interventional trials : 97,396
Observational trials  : 25,694


## 8. Build Cancer Superset -> Subset Hierarchy

This is a presentation-facing structure, so we favor clean cancer labels over maximal recall.

In [16]:
CANCER_KEYWORDS = [
    "cancer", "tumor", "tumour", "carcinoma", "sarcoma", "lymphoma", "leukemia", "leukaemia",
    "melanoma", "neoplasm", "glioma", "glioblastoma", "myeloma", "adenocarcinoma", "blastoma",
    "mesothelioma", "cholangiocarcinoma", "neuroblastoma", "retinoblastoma", "rhabdomyosarcoma",
    "angiosarcoma", "osteosarcoma", "head and neck", "hepatocellular", "myelodysplastic",
]

GENERIC_REMOVE = {
    "cancer", "neoplasm", "neoplasms", "tumor", "tumour", "malignant", "malignancy", "carcinoma",
    "adenocarcinoma", "solid tumor", "solid tumour", "advanced solid tumor", "advanced solid tumour",
    "stage ii", "stoma", "hepatocellular",
}

CANONICAL_RENAMES = {
    "Prostate": "Prostate Cancer",
    "Colorectal": "Colorectal Cancer",
    "Head And Neck": "Head And Neck Cancer",
    "Non-Small Cell Lung": "Non-Small Cell Lung Cancer",
    "Non Small Cell Lung Cancer": "Non-Small Cell Lung Cancer",
    "Cell Lung Cancer": "Lung Cancer",
}

def normalize_text(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text.title()

all_conds = df["conditions"].str.split("|").explode().str.strip()
all_conds = all_conds[(all_conds.notna()) & (all_conds != "")]
cond_counts = all_conds.value_counts()

def is_cancer_term(term: str) -> bool:
    term = str(term).lower()
    return any(k in term for k in CANCER_KEYWORDS)

candidate_terms = cond_counts[cond_counts.index.map(is_cancer_term)]

clean_hierarchy = {}
for cond, count in candidate_terms.items():
    label = normalize_text(cond)
    label = CANONICAL_RENAMES.get(label, label)
    if label.lower() in GENERIC_REMOVE:
        continue
    if count < 3:
        continue
    if label not in clean_hierarchy:
        clean_hierarchy[label] = {"trial_count": int(count), "subtypes": []}
    else:
        clean_hierarchy[label]["trial_count"] = max(clean_hierarchy[label]["trial_count"], int(count))

for broad in list(clean_hierarchy.keys()):
    broad_lower = broad.lower()
    subtype_map = {}
    for cond, count in cond_counts.items():
        norm = normalize_text(cond)
        norm = CANONICAL_RENAMES.get(norm, norm)
        if norm.lower() in GENERIC_REMOVE:
            continue
        if norm.lower() != broad_lower and broad_lower in norm.lower() and count >= 2:
            subtype_map[norm] = max(subtype_map.get(norm, 0), int(count))
    clean_hierarchy[broad]["subtypes"] = [
        {"condition": k, "trial_count": v}
        for k, v in sorted(subtype_map.items(), key=lambda x: x[1], reverse=True)[:30]
    ]

final_hierarchy = dict(sorted(clean_hierarchy.items(), key=lambda x: x[1]["trial_count"], reverse=True)[:75])
final_cancer_df = pd.DataFrame([
    {"cancer": k, "trial_count": v["trial_count"]}
    for k, v in final_hierarchy.items()
]).sort_values("trial_count", ascending=False)

display(final_cancer_df.head(20))

,cancer,trial_count
0,Breast Cancer,8693
1,Prostate Cancer,4388
2,Colorectal Cancer,3421
3,Lung Cancer,3070
4,Multiple Myeloma,2321
5,Lymphoma,2040
6,Ovarian Cancer,2028
7,Pancreatic Cancer,1901
8,Hepatocellular Carcinoma,1799
9,Gastric Cancer,1789


## 9. Title-Derived Features

Title text is cheaper than eligibility text, so we use it aggressively for early structure: diseases, biomarkers, line of therapy, and mono vs combination hints.

In [18]:
title_series = df["official_title"].fillna("").astype(str)
unique_titles = pd.Series(title_series.unique())
unique_titles = unique_titles[unique_titles.ne("")]

print(f"Unique official titles to process: {len(unique_titles):,}")

title_disease_map = {}
for doc in tqdm(nlp_bc5cdr.pipe(unique_titles.tolist(), batch_size=128), total=len(unique_titles), desc="NER on official_title"):
    diseases = sorted({ent.text.strip() for ent in doc.ents if ent.label_ == "DISEASE" and ent.text.strip()})
    title_disease_map[doc.text] = "|".join(diseases)

df["ner_diseases_title"] = title_series.map(title_disease_map).fillna("")

BIOMARKER_MAP_STRICT = {
    "HER2": r"\bHER[-\s]?2\b(?:[\s-]*(?:positive|\+|negative|\-))?\b",
    "EGFR": r"\bEGFR\b(?:[\s-]*(?:mutant|mutation|positive|\+))?\b",
    "BRAF": r"\bBRAF\b(?:[\s-]*V600(?:E|K|D)?)?\b",
    "ALK": r"\bALK\b(?:[\s-]*(?:positive|\+|rearrangement|fusion))?\b",
    "PD-L1": r"\bPD[-\s]?L1\b(?:[\s-]*(?:positive|\+|negative|high|low))?\b",
    "ROS1": r"\bROS1\b(?:[\s-]*(?:positive|\+|rearrangement|fusion))?\b",
    "KRAS": r"\bKRAS\b(?:[\s-]*(?:mutant|mutation|G12[A-Z]))?\b",
    "MSI": r"\bMSI[-\s]?(?:H|HIGH|L|LOW)?\b",
    "dMMR/MMR": r"\bdMMR\b|\bMMR\b",
    "TMB": r"\bTMB\b(?:[\s-]*(?:high|low|H|L))?\b",
    "NTRK": r"\bNTRK\b(?:[\s-]*(?:fusion|positive))?\b",
    "MET": r"\bMET\b(?:[\s-]*(?:amplification|exon\s*14|positive))?\b",
    "FGFR": r"\bFGFR[1-4]?\b(?:[\s-]*(?:mutation|amplification|fusion))?\b",
    "IDH": r"\bIDH[12]?\b(?:[\s-]*(?:mutant|mutation))?\b",
    "PIK3CA": r"\bPIK3CA\b(?:[\s-]*(?:mutant|mutation))?\b",
    "TP53": r"\bTP53\b(?:[\s-]*(?:mutant|mutation))?\b",
    "BRCA": r"\bBRCA[12]?\b(?:[\s-]*(?:mutant|mutation|positive))?\b",
    "CDK4/6": r"\bCDK4/6\b|\bCDK[46]\b(?:[\s-]*(?:amplification))?\b",
    "PDGFR": r"\bPDGFR[AB]?\b(?:[\s-]*(?:mutation|amplification))?\b",
    "VEGF": r"\bVEGF\b",
    "VEGFR": r"\bVEGFR[1-3]?\b(?:[\s-]*(?:positive|overexpression))?\b",
    "mTOR": r"\bmTOR\b(?:[\s-]*(?:mutation|activation))?\b",
    "AR": r"\bAR\b(?:[\s-]*(?:positive|\+|negative|low|high))\b",
    "ER": r"\bER\b(?:[\s-]*(?:positive|\+|negative|low|high))\b",
    "PR": r"\bPR\b(?:[\s-]*(?:positive|\+|negative))\b",
    "CCND1": r"\bCCND1\b(?:[\s-]*(?:amplification))?\b",
    "POLE": r"\bPOLE\b(?:[\s-]*(?:mutant|mutation|exonuclease))?\b",
    "NRG1": r"\bNRG1\b(?:[\s-]*(?:fusion))?\b",
}

def extract_biomarkers_strict(text: str) -> list:
    if pd.isna(text):
        return []
    text = str(text)
    found = []
    for label, pattern in BIOMARKER_MAP_STRICT.items():
        if re.search(pattern, text, flags=re.IGNORECASE):
            found.append(label)
    return sorted(set(found))

LINE_OF_THERAPY_PATTERNS = {
    "Treatment Naive / 1st Line": [r"\b1st[\s-]*line\b", r"\bfirst[\s-]*line\b", r"\btreatment[-\s]*naive\b", r"\buntreated\b", r"\bpreviously untreated\b"],
    "Relapsed / Refractory": [r"\brelapsed\b", r"\brefractory\b", r"\br\/r\b", r"\bfailed prior\b", r"\bafter prior therapy\b"],
    "Adjuvant": [r"\badjuvant\b"],
    "Neoadjuvant": [r"\bneoadjuvant\b", r"\bpreoperative\b"],
    "Maintenance": [r"\bmaintenance\b"],
    "Advanced / Metastatic": [r"\badvanced\b", r"\bmetastatic\b", r"\bunresectable\b", r"\brecurrent\b", r"\bstage iv\b"],
}

def extract_line_of_therapy(text: str) -> str:
    if not text:
        return "Not Specified"
    matched = []
    for label, patterns in LINE_OF_THERAPY_PATTERNS.items():
        for pattern in patterns:
            if re.search(pattern, str(text), flags=re.IGNORECASE):
                matched.append(label)
                break
    if not matched:
        return "Not Specified"
    priority = [
        "Treatment Naive / 1st Line", "Relapsed / Refractory", "Neoadjuvant", "Adjuvant", "Maintenance", "Advanced / Metastatic"
    ]
    for label in priority:
        if label in matched:
            return label
    return matched[0]

def is_combination_therapy(text: str) -> bool:
    if not text:
        return False
    strong_combo_patterns = [
        r"\bcombination\b", r"\bcombined with\b", r"\bin combination with\b", r"\bplus\b", r"\bplus placebo\b", r"\b[A-Za-z0-9-]+\s*\+\s*[A-Za-z0-9-]+"
    ]
    return any(re.search(pat, str(text), flags=re.IGNORECASE) for pat in strong_combo_patterns)

df["title_biomarkers"] = title_series.map(extract_biomarkers_strict)
df["title_biomarkers_str"] = df["title_biomarkers"].map(lambda vals: "|".join(vals))
df["line_of_therapy"] = title_series.map(extract_line_of_therapy)
df["is_combination"] = title_series.map(is_combination_therapy)

print(f"Trials with biomarker in title : {(df['title_biomarkers'].map(len) > 0).sum():,}")
print("\nTop biomarkers in titles:")
display(df["title_biomarkers"].explode().dropna().value_counts().head(20).to_frame("trial_mentions"))
print("\nLine of therapy distribution:")
print(df["line_of_therapy"].value_counts().to_string())
print("\nCombination vs mono:")
print(df["is_combination"].value_counts().to_string())

Unique official titles to process: 121,913


NER on official_title:   0%|          | 0/121913 [00:00<?, ?it/s]

Trials with biomarker in title : 7,944

Top biomarkers in titles:


,trial_mentions
title_biomarkers,
HER2,3132
EGFR,1371
PD-L1,817
KRAS,433
BRAF,407
BRCA,296
ALK,286
VEGF,233
MET,224



Line of therapy distribution:
line_of_therapy
Not Specified                 74434
Advanced / Metastatic         26736
Relapsed / Refractory          8597
Treatment Naive / 1st Line     5138
Neoadjuvant                    4929
Adjuvant                       2658
Maintenance                    1034

Combination vs mono:
is_combination
False    100699
True      22827


## 10. Eligibility-Derived Features

This is the longest block in the notebook. It uses `nlp.pipe()` on unique interventional eligibility texts and shows exact progress.

In [19]:
def clean_eligibility_text(text, max_chars=2000):
    if pd.isna(text):
        return ""
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text[:max_chars]

elig_mask = (
    df["study_type"].eq("INTERVENTIONAL")
    & df["eligibility_criteria"].fillna("").astype(str).str.strip().ne("")
)

df["eligibility_text_for_ner"] = ""
df.loc[elig_mask, "eligibility_text_for_ner"] = df.loc[elig_mask, "eligibility_criteria"].map(clean_eligibility_text)

unique_eligibility = pd.Series(df.loc[elig_mask, "eligibility_text_for_ner"].unique())
unique_eligibility = unique_eligibility[unique_eligibility.ne("")]

print(f"Interventional rows with eligibility text : {elig_mask.sum():,}")
print(f"Unique eligibility texts to process       : {len(unique_eligibility):,}")

chem_map = {}
disease_map = {}
for doc in tqdm(nlp_bc5cdr.pipe(unique_eligibility.tolist(), batch_size=64), total=len(unique_eligibility), desc="Eligibility NER"):
    chemicals = sorted({ent.text.strip() for ent in doc.ents if ent.label_ == "CHEMICAL" and ent.text.strip()})
    diseases = sorted({ent.text.strip() for ent in doc.ents if ent.label_ == "DISEASE" and ent.text.strip()})
    chem_map[doc.text] = "|".join(chemicals)
    disease_map[doc.text] = "|".join(diseases)

df["ner_chemicals_eligibility"] = ""
df["ner_diseases_eligibility"] = ""
df.loc[elig_mask, "ner_chemicals_eligibility"] = df.loc[elig_mask, "eligibility_text_for_ner"].map(chem_map).fillna("")
df.loc[elig_mask, "ner_diseases_eligibility"] = df.loc[elig_mask, "eligibility_text_for_ner"].map(disease_map).fillna("")

print("Eligibility NER completed.")
display(df[["ner_chemicals_eligibility", "ner_diseases_eligibility"]].head())

Interventional rows with eligibility text : 97,389
Unique eligibility texts to process       : 96,999


Eligibility NER:   0%|          | 0/96999 [00:00<?, ?it/s]

Eligibility NER completed.


,ner_chemicals_eligibility,ner_diseases_eligibility
0,ALK|RECIST|corticosteroids,ALK-positive non-small cell lung cancer|NSCLC|allergy/hypersensitivity|cancer
1,resistant/refractory|status ≤ 2 4,Primary Brain Tumors|Proteinuria|cardiac impairment|gastrointestinal malabsorption|myocardial infarction|squamous cell carcinoma of the lung 5|toxicities|tu...
2,,
3,RECIST|corticosteroids|prednisone|steroid|steroids,Tumor|Tumors|adenocarcinoma of the stomach or gastro-esophageal junction|adrenal insufficiency
4,,


In [21]:
def combine_pipe(*args):
    all_items = []
    for arg in args:
        if pd.notna(arg) and str(arg).strip() and str(arg).strip().lower() != "nan":
            all_items.extend([x.strip() for x in str(arg).split("|") if x.strip()])
    return "|".join(list(dict.fromkeys(all_items)))

df["biomarkers_from_eligibility"] = df["eligibility_criteria"].fillna("").astype(str).map(lambda x: "|".join(extract_biomarkers_strict(x)))
df["biomarkers_all"] = df.apply(lambda row: combine_pipe(row["title_biomarkers_str"], row["biomarkers_from_eligibility"]), axis=1)

MUTATION_MAP = {
    "BRAF V600": r"\bBRAF\b[\s-]*V600[A-Z]?\b",
    "EGFR": r"\bEGFR\b(?:[\s-]*(?:exon\s*\d+|L858R|T790M|C797S|del\s*19))?\b",
    "KRAS": r"\bKRAS\b(?:[\s-]*(?:G12[A-Z]|G13[A-Z]|Q61[A-Z]))?\b",
    "NRAS": r"\bNRAS\b(?:[\s-]*(?:Q61[A-Z]|G12[A-Z]))?\b",
    "ALK": r"\bALK\b(?:[\s-]*(?:fusion|rearrangement|F1174L|R1275Q))?\b",
    "ROS1": r"\bROS1\b(?:[\s-]*(?:fusion|rearrangement))?\b",
    "MET": r"\bMET\b(?:[\s-]*(?:exon\s*14|amplification|Y1253D))?\b",
    "PIK3CA": r"\bPIK3CA\b(?:[\s-]*(?:E545K|H1047R|E542K))?\b",
    "IDH1/2": r"\bIDH[12]\b(?:[\s-]*(?:R132[A-Z]|R140Q|R172[A-Z]))?\b",
    "FGFR1-4": r"\bFGFR[1-4]\b(?:[\s-]*(?:fusion|mutation|amplification))?\b",
    "RET": r"\bRET\b(?:[\s-]*(?:fusion|M918T|C634[A-Z]))?\b",
    "BRCA1/2": r"\bBRCA[12]\b(?:[\s-]*(?:mutation|del|ins))?\b",
    "TP53": r"\bTP53\b(?:[\s-]*(?:R175H|R248[A-Z]|R273[A-Z]))?\b",
    "PTEN": r"\bPTEN\b(?:[\s-]*(?:loss|mutation|deletion))?\b",
    "CDK4/6": r"\bCDK4/6\b",
    "HER2": r"\bHER[-\s]?2\b(?:[\s-]*(?:amplification|overexpression|mutation|positive|\+))?\b",
    "JAK1/2": r"\bJAK[12]\b(?:[\s-]*(?:mutation|V617F|exon\s*12))?\b",
    "NPM1": r"\bNPM1\b(?:[\s-]*(?:mutant|mutation|exon\s*12))?\b",
    "FLT3": r"\bFLT3\b(?:[\s-]*(?:ITD|TKD|mutation|D835[A-Z]?))?\b",
    "BCR-ABL": r"\bBCR[-\s]?ABL\b(?:[\s-]*(?:T315I|fusion))?\b",
    "ABL1": r"\bABL1\b(?:[\s-]*(?:T315I|mutation))?\b",
    "DNMT3A": r"\bDNMT3A\b(?:[\s-]*(?:R882[A-Z]?|mutant|mutation))?\b",
    "ASXL1": r"\bASXL1\b(?:[\s-]*(?:mutant|mutation))?\b",
    "TET2": r"\bTET2\b(?:[\s-]*(?:mutant|mutation))?\b",
    "SF3B1": r"\bSF3B1\b(?:[\s-]*(?:K700E|mutant|mutation))?\b",
    "NOTCH1": r"\bNOTCH1\b(?:[\s-]*(?:mutant|mutation|activation))?\b",
    "CEBPA": r"\bCEBPA\b(?:[\s-]*(?:mutant|mutation|biallelic))?\b",
    "KIT": r"\bKIT\b(?:[\s-]*(?:D816V|exon\s*11|exon\s*9|mutation))?\b",
    "PDGFRA": r"\bPDGFRA\b(?:[\s-]*(?:D842V|mutation|exon\s*18))?\b",
    "NF1": r"\bNF1\b(?:[\s-]*(?:mutation|loss|deletion))?\b",
    "VHL": r"\bVHL\b(?:[\s-]*(?:mutation|loss|deletion))?\b",
    "SDH": r"\bSDH[ABCD]?\b(?:[\s-]*(?:mutation|loss|deficient))?\b",
}

def extract_mutations_strict(text: str) -> list:
    if pd.isna(text):
        return []
    text = str(text)
    found = []
    for label, pattern in MUTATION_MAP.items():
        if re.search(pattern, text, flags=re.IGNORECASE):
            found.append(label)
    return sorted(set(found))

df["mutations_from_title"] = df["official_title"].fillna("").astype(str).map(lambda x: "|".join(extract_mutations_strict(x)))
df["mutations_from_eligibility"] = df["eligibility_criteria"].fillna("").astype(str).map(lambda x: "|".join(extract_mutations_strict(x)))
df["mutations_all"] = df.apply(lambda row: combine_pipe(row["mutations_from_title"], row["mutations_from_eligibility"]), axis=1)

BIOMARKER_LEVEL_PATTERNS = {
    "PD-L1 %": r"\bPD[-\s]?L1\b[^\n\r;,.]{0,40}?(>=|<=|>|<|=)?\s*\d+\s*%",
    "CPS": r"\bCPS\b\s*(>=|<=|>|<|=)?\s*\d+",
    "TPS": r"\bTPS\b\s*(>=|<=|>|<|=)?\s*\d+",
    "TMB": r"\bTMB\b[^\n\r;,.]{0,30}?(>=|<=|>|<|=)?\s*\d+",
}

PATIENT_PROFILE_PATTERNS = {
    "Treatment Naive": [r"\btreatment[-\s]*naive\b", r"\bpreviously untreated\b", r"\buntreated\b"],
    "Prior Therapy": [r"\bprior therapy\b", r"\bprevious therapy\b", r"\bpreviously treated\b", r"\bafter prior\b"],
    "Relapsed/Refractory": [r"\brelapsed\b", r"\brefractory\b", r"\br\/r\b"],
    "Advanced/Metastatic": [r"\badvanced\b", r"\bmetastatic\b", r"\bunresectable\b", r"\brecurrent\b"],
}

def extract_biomarker_levels(text: str) -> list:
    if pd.isna(text):
        return []
    text = str(text)
    found = []
    for label, pattern in BIOMARKER_LEVEL_PATTERNS.items():
        if re.search(pattern, text, flags=re.IGNORECASE):
            found.append(label)
    return sorted(set(found))

def extract_patient_profile_flags(text: str) -> list:
    if pd.isna(text):
        return []
    text = str(text)
    found = []
    for label, patterns in PATIENT_PROFILE_PATTERNS.items():
        for pattern in patterns:
            if re.search(pattern, text, flags=re.IGNORECASE):
                found.append(label)
                break
    return sorted(set(found))

df["biomarker_levels_eligibility"] = df["eligibility_criteria"].fillna("").astype(str).map(lambda x: "|".join(extract_biomarker_levels(x)))
df["patient_profile_flags"] = df["eligibility_criteria"].fillna("").astype(str).map(lambda x: "|".join(extract_patient_profile_flags(x)))

print(f"Trials with any biomarker        : {(df['biomarkers_all'] != '').sum():,}")
print(f"Trials with mutation info        : {(df['mutations_all'] != '').sum():,}")
print(f"Trials with biomarker levels     : {(df['biomarker_levels_eligibility'] != '').sum():,}")
print(f"Trials with patient profile flag : {(df['patient_profile_flags'] != '').sum():,}")

print("\nTop biomarkers overall:")
display(df["biomarkers_all"].str.split("|").explode().dropna().loc[lambda s: s != ""].value_counts().head(20).to_frame("trial_mentions"))
print("\nTop mutations overall:")
display(df["mutations_all"].str.split("|").explode().dropna().loc[lambda s: s != ""].value_counts().head(20).to_frame("trial_mentions"))

Trials with any biomarker        : 26,987
Trials with mutation info        : 21,120
Trials with biomarker levels     : 407
Trials with patient profile flag : 62,006

Top biomarkers overall:


,trial_mentions
biomarkers_all,
PD-L1,7120
EGFR,6735
HER2,6726
MET,4529
ALK,2493
BRAF,1765
VEGF,1675
BRCA,1318
KRAS,1175



Top mutations overall:


,trial_mentions
mutations_all,
EGFR,6735
HER2,6726
MET,4529
ALK,2493
KRAS,1175
ROS1,857
BRCA1/2,814
CDK4/6,753
BRAF V600,635


## 11. Outcome Standardization

In [22]:
OUTCOME_MAP = {
    "Overall Survival": [r"\bOS\b", r"overall[\s]*survival", r"overall[\s]*surviv"],
    "Progression Free Survival": [r"\bPFS\b", r"progression[\s]*free[\s]*survival", r"progression[\s]*free"],
    "Overall Response Rate": [r"\bORR\b", r"overall[\s]*response[\s]*rate", r"objective[\s]*response[\s]*rate"],
    "Complete Response Rate": [r"\bCR\b", r"complete[\s]*response", r"complete[\s]*remission"],
    "Disease Control Rate": [r"\bDCR\b", r"disease[\s]*control[\s]*rate"],
    "Duration of Response": [r"\bDOR\b", r"duration[\s]*of[\s]*response"],
    "Time to Progression": [r"\bTTP\b", r"time[\s]*to[\s]*progression"],
    "Event Free Survival": [r"\bEFS\b", r"event[\s]*free[\s]*survival"],
    "Pathological Complete Response": [r"\bpCR\b", r"pathological[\s]*complete[\s]*response"],
    "Minimal Residual Disease": [r"\bMRD\b", r"minimal[\s]*residual[\s]*disease"],
    "Clinical Benefit Rate": [r"\bCBR\b", r"clinical[\s]*benefit[\s]*rate"],
    "Relapse Free Survival": [r"\bRFS\b", r"relapse[\s]*free[\s]*survival"],
    "Cancer Specific Survival": [r"\bCSS\b", r"cancer[\s]*specific[\s]*survival"],
    "Distant Metastasis Free": [r"\bDMFS\b", r"distant[\s]*metastasis[\s]*free"],
    "Biochemical Recurrence": [r"\bBCR\b", r"biochemical[\s]*recurrence", r"PSA[\s]*(progression|recurrence)"],
    "Safety and Tolerability": [r"safety[\s]*and[\s]*tolerability", r"dose[\s]*limiting[\s]*toxicity", r"\bDLT\b", r"maximum[\s]*tolerated[\s]*dose", r"\bMTD\b"],
}

def standardize_outcomes(outcome_str: str) -> str:
    if not outcome_str or str(outcome_str).strip().lower() == "nan":
        return ""
    found = []
    text = str(outcome_str)
    for standard_name, patterns in OUTCOME_MAP.items():
        for pattern in patterns:
            if re.search(pattern, text, flags=re.IGNORECASE):
                found.append(standard_name)
                break
    return "|".join(list(dict.fromkeys(found)))

df["primary_outcomes_standardized"] = df["primary_outcome_measures"].apply(standardize_outcomes)

print(f"Trials with standardized outcomes: {(df['primary_outcomes_standardized'] != '').sum():,}")
display(df["primary_outcomes_standardized"].str.split("|").explode().dropna().loc[lambda s: s != ""].value_counts().to_frame("trial_mentions").head(20))

Trials with standardized outcomes: 37,817


,trial_mentions
primary_outcomes_standardized,
Safety and Tolerability,11115
Overall Response Rate,9480
Progression Free Survival,7891
Overall Survival,6186
Complete Response Rate,4881
Pathological Complete Response,1515
Event Free Survival,756
Disease Control Rate,653
Time to Progression,545


## 12. Asset Normalization and Non-Drug Filtering

This is the section that prevents `Placebo`, radiotherapy strings, and broad therapy labels from leaking into the asset layer.

In [23]:
df_inv_drugs = df_inv[df_inv["intervention_type"].isin(["DRUG", "BIOLOGICAL"])].copy()

alias_to_canonical = {}
for _, row in df_inv_drugs.iterrows():
    main_name = str(row["intervention_name"]).strip()
    other_names = str(row["other_names"]).strip()
    if not main_name or main_name.lower() == "nan":
        continue
    aliases = [main_name]
    if other_names and other_names.lower() != "nan":
        aliases.extend([name.strip() for name in other_names.split("|") if name.strip()])
    for alias in aliases:
        alias_to_canonical.setdefault(alias.lower(), main_name)

print(f"Total alias mappings: {len(alias_to_canonical):,}")

Total alias mappings: 66,735


In [24]:
NON_DRUG_PATTERNS = [
    r"\bplacebo\b",
    r"\bvehicle\b",
    r"\bcontrol\b",
    r"\bbest supportive care\b",
    r"\bsupportive care\b",
    r"\bstandard of care\b",
    r"\bobservation\b",
    r"\bwatchful waiting\b",
    r"\bradiotherapy\b",
    r"\bradiation\b",
    r"\bmultimodality radiotherapy\b",
    r"\bsurgery\b",
    r"\bsurgical\b",
    r"\bresection\b",
    r"\bprocedure\b",
    r"\btransplant\b",
    r"\bstem cell transplant\b",
    r"\bchemotherapy\b$",
    r"\bendocrine therapy\b",
    r"\bhormone therapy\b",
    r"\bimaging\b",
    r"\bscan\b",
    r"\bbiopsy\b",
    r"\bsample\b",
    r"\bsamples\b",
    r"\banalysis\b",
    r"\bexercise\b",
    r"\bdiet\b",
    r"\bfollow-up\b",
    r"\bfollow up\b",
    r"\bquestionnaire\b",
    r"\bphysician'?s choice\b",
    r"\binvestigator'?s choice\b",
]

def normalize_drug(name: str) -> str:
    raw = str(name).strip()
    if not raw or raw.lower() == "nan":
        return ""
    return alias_to_canonical.get(raw.lower(), raw)

def is_non_drug_asset(name: str) -> bool:
    text = str(name).strip().lower()
    if not text or text == "nan":
        return True
    for pattern in NON_DRUG_PATTERNS:
        if re.search(pattern, text, flags=re.IGNORECASE):
            return True
    if " or " in text:
        return True
    if " regimen" in text or " therapy" in text:
        allow_terms = ["car-t", "cart", "vaccine", "antibody", "inhibitor", "conjugate"]
        if not any(term in text for term in allow_terms):
            return True
    return False

def clean_asset_name(name: str) -> str:
    text = str(name).strip()
    if not text or text.lower() == "nan":
        return ""
    text = re.sub(r"\s+", " ", text).strip()
    if text.islower():
        text = text.title()
    return text

def normalize_drug_list(pipe_str: str) -> str:
    if not pipe_str or str(pipe_str).strip().lower() == "nan":
        return ""
    parts = re.split(r"\|", str(pipe_str))
    candidates = []
    for part in parts:
        subparts = re.split(r"\s*\+\s*", part)
        for sub in subparts:
            sub = sub.strip()
            if not sub:
                continue
            normalized = normalize_drug(sub)
            cleaned = clean_asset_name(normalized)
            if not cleaned:
                continue
            if is_non_drug_asset(cleaned):
                continue
            candidates.append(cleaned)

    seen_lower = set()
    final_assets = []
    for asset in candidates:
        if asset.lower() not in seen_lower:
            seen_lower.add(asset.lower())
            final_assets.append(asset)
    return "|".join(final_assets)

df["all_drugs_normalized"] = df["all_drugs_raw"].apply(normalize_drug_list)

print("Drug normalization + non-drug filtering applied.")
changed = (df["all_drugs_raw"] != df["all_drugs_normalized"]).sum()
print(f"Trials where asset labels changed : {changed:,}")
placebo_rows = df[df["all_drugs_normalized"].str.contains("placebo", case=False, na=False)]
print(f"Rows still containing placebo     : {len(placebo_rows):,}")

display(
    df[df["all_drugs_raw"] != df["all_drugs_normalized"]][
        ["nct_id", "all_drugs_raw", "all_drugs_normalized"]
    ].head(15)
)

df["all_drugs_normalized_list"] = df["all_drugs_normalized"].map(parse_pipe)

Drug normalization + non-drug filtering applied.
Trials where asset labels changed : 28,602
Rows still containing placebo     : 61


,nct_id,all_drugs_raw,all_drugs_normalized
2,NCT07490990,Disitamab vedotin + Sintilimab + Multimodality radiotherapy,Disitamab Vedotin|Sintilimab
9,NCT01924260,alisertib|gemcitabine,Alisertib|Gemcitabine
11,NCT00003220,bryostatin 1,Bryostatin 1
26,NCT00004688,carmustine|mercaptopurine|streptozocin,Carmustine|Mercaptopurine|Streptozocin
32,NCT04031703,Paclitaxel|Carboplatin|Epirubicin|Cyclophosphamide|5-fluorouracil|Docetaxel,Paclitaxel|Carboplatin|Epirubicin|Cyclophosphamide|5-Fluorouracil|Docetaxel
46,NCT01387269,Anamorelin HCl|Placebo,Anamorelin HCl
49,NCT02592083,Tamoxifen or Aromatase Inhibitor or Aromatase Inhibitor + goserelin|Palbociclib,Goserelin|Palbociclib
56,NCT01093183,lenalidomide|cyclophosphamide,Lenalidomide|Cyclophosphamide
58,NCT00376883,pamidronate,Pamidronate
66,NCT01911325,Buparlisib|Buparlisib matching placebo|Docetaxel,Buparlisib|Docetaxel


## 13. Combination Partners and Drug-Phase Mapping

In [25]:
drug_partners = defaultdict(set)
for _, row in tqdm(df.iterrows(), total=len(df), desc="Building drug partner map"):
    drugs = row["all_drugs_normalized_list"] if isinstance(row["all_drugs_normalized_list"], list) else []
    if len(drugs) > 1:
        for drug in drugs:
            partners = [d for d in drugs if d.lower() != drug.lower()]
            drug_partners[drug.lower()].update(partners)

print(f"Drugs with combination partner data: {len(drug_partners):,}")
for drug, partners in list(drug_partners.items())[:5]:
    print(f"  {drug} -> {list(partners)[:5]}")

all_drug_entries = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Building drug-phase map"):
    drugs = row["all_drugs_normalized_list"] if isinstance(row["all_drugs_normalized_list"], list) else []
    buckets = row["phase_buckets"] if isinstance(row["phase_buckets"], list) else []
    for drug in drugs:
        for bucket in (buckets if buckets else ["NA"]):
            all_drug_entries.append({
                "drug_name": drug,
                "phase_bucket": bucket,
                "nct_id": row["nct_id"],
                "overall_status": row["overall_status"],
                "conditions": row["conditions"],
                "study_type": row["study_type"],
            })

df_drug_phase = pd.DataFrame(all_drug_entries)
display(df_drug_phase.head())
print(f"\nDrug-phase rows: {len(df_drug_phase):,}")
print("Unique assets per phase:")
print(df_drug_phase.groupby("phase_bucket")["drug_name"].nunique().to_string())

Building drug partner map:   0%|          | 0/123526 [00:00<?, ?it/s]

Drugs with combination partner data: 19,409
  disitamab vedotin -> ['Disitamab Vedotin Tislelizumab', 'Lactobacillus Plantarum IS-10506', 'Tucatinib', 'S1', 'Cadonilimab']
  sintilimab -> ['Cabozantinib', 'Paclitaxel/Albumin-Bound Paclitaxel', 'Donafenib', 'Tegafur-Gimeracil-Oteracil', 'IBI310']
  avelumab -> ['Cabozantinib', 'Aflibercept', 'Glasdegib Maleate', 'Lansoprazole', 'PD-L1 t-haNK']
  oxaliplatin -> ['Cabozantinib', 'Ziv-Aflibercept', 'Tucatinib', 'Cemiplimab', 'Tegafur gimeracil oteracil potassium capsule']
  5-fluorouracil -> ['Zolbetuximab', 'Aflibercept', 'Irinotecan Injection [Camptosar]', 'CMAB009', 'TAS-102']


Building drug-phase map:   0%|          | 0/123526 [00:00<?, ?it/s]

,drug_name,phase_bucket,nct_id,overall_status,conditions,study_type
0,TRI-611,Phase 1,NCT07491497,RECRUITING,ALK-positive NSCLC|ALK-Positive Lung Cancer|ALK-positive Non-small Cell Lung Cancer,INTERVENTIONAL
1,TRI-611,Phase 2,NCT07491497,RECRUITING,ALK-positive NSCLC|ALK-Positive Lung Cancer|ALK-positive Non-small Cell Lung Cancer,INTERVENTIONAL
2,TKI258,Phase 1,NCT00669097,COMPLETED,Advanced Solid Malignancies,INTERVENTIONAL
3,Disitamab Vedotin,NA,NCT07490990,NOT_YET_RECRUITING,HER2-positive Advanced Gastric Cancer or Gastroesophageal Junction Adenocarcinoma,OBSERVATIONAL
4,Sintilimab,NA,NCT07490990,NOT_YET_RECRUITING,HER2-positive Advanced Gastric Cancer or Gastroesophageal Junction Adenocarcinoma,OBSERVATIONAL



Drug-phase rows: 144,892
Unique assets per phase:
phase_bucket
Early Phase 1     1385
NA                4093
Phase 1          14360
Phase 2          16808
Phase 3           6258
Phase 4           2010


## 14. Save Processed Files

In [26]:
with open(processed_dir / "cancer_hierarchy_final.json", "w", encoding="utf-8") as f:
    json.dump(final_hierarchy, f, indent=2, ensure_ascii=False)

final_cancer_df.to_csv(processed_dir / "cancer_conditions_list_final.csv", index=False)
df.to_csv(processed_dir / "trials_processed.csv", index=False, encoding="utf-8-sig")
df_inv_drugs.to_csv(processed_dir / "interventions_processed.csv", index=False, encoding="utf-8-sig")
df_drug_phase.to_csv(processed_dir / "drug_phase_map.csv", index=False, encoding="utf-8-sig")

drug_partners_serializable = {k: sorted(list(v)) for k, v in drug_partners.items()}
with open(processed_dir / "drug_combination_partners.json", "w", encoding="utf-8") as f:
    json.dump(drug_partners_serializable, f, indent=2, ensure_ascii=False)

print("Files saved:")
for fpath in [
    processed_dir / "trials_processed.csv",
    processed_dir / "interventions_processed.csv",
    processed_dir / "drug_phase_map.csv",
    processed_dir / "cancer_hierarchy_final.json",
    processed_dir / "cancer_conditions_list_final.csv",
    processed_dir / "drug_combination_partners.json",
]:
    mb = fpath.stat().st_size / (1024 ** 2)
    print(f"  {mb:7.1f} MB -> {fpath}")

Files saved:
    651.1 MB -> C:\Users\Lakshay\Desktop\Machine_Learning\Clinical Trials\data\processed\trials_processed.csv
     41.4 MB -> C:\Users\Lakshay\Desktop\Machine_Learning\Clinical Trials\data\processed\interventions_processed.csv
     18.1 MB -> C:\Users\Lakshay\Desktop\Machine_Learning\Clinical Trials\data\processed\drug_phase_map.csv
      0.2 MB -> C:\Users\Lakshay\Desktop\Machine_Learning\Clinical Trials\data\processed\cancer_hierarchy_final.json
      0.0 MB -> C:\Users\Lakshay\Desktop\Machine_Learning\Clinical Trials\data\processed\cancer_conditions_list_final.csv
      3.8 MB -> C:\Users\Lakshay\Desktop\Machine_Learning\Clinical Trials\data\processed\drug_combination_partners.json


## 15. Final Summary

In [27]:
div = "=" * 70
print(div)
print("EDA + FEATURE ENGINEERING SUMMARY")
print(div)
print(f"Total trials                     : {len(df):,}")
print(f"Columns after pruning            : {len(df.columns)}")
print()
print("Study type:")
print(df["study_type"].value_counts().to_string())
print()
print("Phase funnel:")
phase_counts = pd.Series([b for buckets in df["phase_buckets"] for b in buckets])
print(phase_counts.value_counts().reindex(PHASE_ORDER).fillna(0).astype(int).to_string())
print()
print("NER / extraction results:")
print(f"  Diseases from title            : {(df['ner_diseases_title'] != '').sum():,}")
print(f"  Chemicals from eligibility     : {(df['ner_chemicals_eligibility'] != '').sum():,}")
print(f"  Biomarkers extracted           : {(df['biomarkers_all'] != '').sum():,}")
print(f"  Mutations extracted            : {(df['mutations_all'] != '').sum():,}")
print(f"  Biomarker levels found         : {(df['biomarker_levels_eligibility'] != '').sum():,}")
print(f"  Patient profile flags          : {(df['patient_profile_flags'] != '').sum():,}")
print(f"  Outcomes standardized          : {(df['primary_outcomes_standardized'] != '').sum():,}")
print()
print("Drug intelligence layer:")
print(f"  Trials with normalized assets  : {(df['all_drugs_normalized'] != '').sum():,}")
print(f"  Unique normalized assets       : {df_drug_phase['drug_name'].nunique():,}")
print(f"  Assets with partner data       : {len(drug_partners):,}")
print()
print("Line of therapy:")
print(df["line_of_therapy"].value_counts().to_string())
print()
print("Combination vs Mono:")
print(df["is_combination"].value_counts().to_string())
print(div)
print("Next -> 03_visualizations.ipynb")

EDA + FEATURE ENGINEERING SUMMARY
Total trials                     : 123,526
Columns after pruning            : 69

Study type:
study_type
INTERVENTIONAL     97396
OBSERVATIONAL      25694
EXPANDED_ACCESS      436

Phase funnel:
Early Phase 1     1942
Phase 1          25169
Phase 2          40266
Phase 3          11105
Phase 4           2635

NER / extraction results:
  Diseases from title            : 101,081
  Chemicals from eligibility     : 74,060
  Biomarkers extracted           : 26,987
  Mutations extracted            : 21,120
  Biomarker levels found         : 407
  Patient profile flags          : 62,006
  Outcomes standardized          : 37,817

Drug intelligence layer:
  Trials with normalized assets  : 64,875
  Unique normalized assets       : 32,004
  Assets with partner data       : 19,409

Line of therapy:
line_of_therapy
Not Specified                 74434
Advanced / Metastatic         26736
Relapsed / Refractory          8597
Treatment Naive / 1st Line     5138
Neoadju